<a href="https://colab.research.google.com/github/burakderee/siir-olusturucu/blob/main/arayuz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit unsloth -q
!npm install -g localtunnel 2>/dev/null
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile app.py

import streamlit as st
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

st.set_page_config(
    page_title="Orhan Veli Şiir Üretici",
    page_icon="📜",
    layout="wide"
)

st.title("📜 Orhan Veli Tarzı Şiir Üretici")
st.markdown("*Garip Akımı üslubunda, modern kelimelerle şiir üretimi*")
st.divider()

@st.cache_resource
def model_yukle(model_adi):
    if model_adi == "LLaMA-3 8B 🏆":
        from unsloth import FastLanguageModel
        from peft import PeftModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
            max_seq_length=1024,
            load_in_4bit=True,
            dtype=None,
        )
        model = PeftModel.from_pretrained(
            model,
            "/content/drive/MyDrive/llama3_orhan_veli_v2"
        )
        FastLanguageModel.for_inference(model)
        return model, tokenizer

    elif model_adi == "GPT-2 Türkçe":
        tokenizer = AutoTokenizer.from_pretrained(
            "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
        )
        model = AutoModelForCausalLM.from_pretrained(
            "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
        )
        if torch.cuda.is_available():
            model = model.cuda()
        return model, tokenizer

    elif model_adi == "Qwen2.5 1.5B":
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="/content/drive/MyDrive/qwen_orhan_veli_modeli",  # ✅ fine-tuned
            max_seq_length=1024,
            load_in_4bit=True,
            dtype=None,
        )
        FastLanguageModel.for_inference(model)
        return model, tokenizer

def siir_uret(model, tokenizer, kelime, model_adi, max_deneme=3):
    if "LLaMA" in model_adi:
        prompt = """### Talimat:
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
YASAK: kafiye, dramatik dil.
ZORUNLU: "{kelime}" kelimesi şiirde geçmeli.

### Anahtar Kelime:
{kelime}

### Şiir (içinde "{kelime}" geçmeli):
""".format(kelime=kelime)
        inputs = tokenizer([prompt], return_tensors="pt")

    elif model_adi == "GPT-2 Türkçe":
        prompt = f"### Talimat:\nSen Orhan Veli Kanık'sın. \"{kelime}\" kelimesini kullanarak şiir yaz.\n\n### Şiir:\n"
        inputs = tokenizer(prompt, return_tensors="pt")

    elif "Qwen" in model_adi:
        prompt = f"""<|im_start|>system
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
YASAK: HTML, kafiye.
ZORUNLU: "{kelime}" kelimesi şiirde geçmeli.<|im_end|>
<|im_start|>user
"{kelime}" kelimesini kullanarak şiir yaz.<|im_end|>
<|im_start|>assistant
"""
        inputs = tokenizer([prompt], return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    for deneme in range(1, max_deneme + 1):
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=120,
                temperature=0.7 + (deneme * 0.1),
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        siir = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        siir = re.sub(r'<[^>]+>', '', siir)
        dizeler = [d for d in siir.split('\n') if d.strip()]
        siir = '\n'.join(dizeler[:8])

        if kelime.lower() in siir.lower():
            return siir

    # Son çare — manuel yerleştir
    dizeler = [d for d in siir.split('\n') if d.strip()]
    if dizeler:
        dizeler.insert(1, f"{kelime.capitalize()} diyorlar buna")
    return '\n'.join(dizeler[:8])

# ── SIDEBAR ───────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Ayarlar")

    model_secimi = st.selectbox(
        "Model Seç:",
        ["LLaMA-3 8B 🏆", "GPT-2 Türkçe", "Qwen2.5 1.5B"],
        index=0
    )

    st.divider()
    st.subheader("📊 Model Metrikleri")

    metrikler = {
        "LLaMA-3 8B 🏆": {"KKO": "%100.0 🏆", "Perplexity": "28.59", "BLEU": "0.0052"},
        "GPT-2 Türkçe":  {"KKO": "%85.0",     "Perplexity": "28.69", "BLEU": "0.0083 🏆"},
        "Qwen2.5 1.5B":  {"KKO": "%50.0",     "Perplexity": "21.17 🏆", "BLEU": "0.0053"},
    }

    secili = metrikler[model_secimi]
    st.metric("Kelime Kullanım Oranı", secili["KKO"])
    st.metric("Perplexity", secili["Perplexity"])
    st.metric("BLEU Skoru", secili["BLEU"])

    st.divider()
    st.subheader("ℹ️ Model Hakkında")

    aciklamalar = {
        "LLaMA-3 8B 🏆": "Meta'nın 8B parametreli modeli. En yüksek kelime kullanım oranı (%100).",
        "GPT-2 Türkçe":  "YTÜ Türkçe GPT-2. Küçük ama etkili, en yüksek BLEU skoru.",
        "Qwen2.5 1.5B":  "Alibaba'nın 1.5B modeli. En düşük perplexity, hafif model.",
    }
    st.info(aciklamalar[model_secimi])
    st.caption("Doğal Dil İşleme Projesi — 2026")

# ── ANA ALAN ──────────────────────────────────────────────────
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("🎯 Şiir Üret")

    kelime = st.text_input(
        "Anahtar kelime gir:",
        placeholder="wifi, suşi, bitcoin, trip...",
        max_chars=50
    )

    uret_btn = st.button("✍️ Şiir Yaz", type="primary", use_container_width=True)

    if uret_btn and kelime:
        with st.spinner(f"{model_secimi} şiir yazıyor..."):
            try:
                model, tokenizer = model_yukle(model_secimi)
                siir = siir_uret(model, tokenizer, kelime, model_secimi)
                kelime_var = kelime.lower() in siir.lower()

                st.success("Şiir üretildi!")
                st.text_area("📝 Şiir:", siir, height=200)

                if kelime_var:
                    st.success(f"✅ '{kelime}' kelimesi şiirde kullanıldı.")
                else:
                    st.warning(f"⚠️ '{kelime}' kelimesi şiirde bulunamadı.")

            except Exception as e:
                st.error(f"Hata: {e}")

    elif uret_btn and not kelime:
        st.warning("Lütfen bir kelime gir.")

with col2:
    st.subheader("📊 Model Karşılaştırması")
    try:
        st.image(
            "/content/drive/MyDrive/model_karsilastirma_final.png",
            caption="3 Model Performans Karşılaştırması",
            use_container_width=True
        )
    except:
        st.info("Karşılaştırma grafiği bulunamadı.")

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok
import subprocess, threading, time

# BURAYA kendi token'ını yapıştır
ngrok.set_auth_token("KENDI_NGROK_TOKENINIZI_BURAYA_YAZIN")
def streamlit_calistir():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

thread = threading.Thread(target=streamlit_calistir)
thread.daemon = True
thread.start()

time.sleep(5)

public_url = ngrok.connect(8501)
print(f"\n✅ Uygulama açık: {public_url}")